# Bhagavad Gita -> chanted audio (Vāgdhenu TTS) - Kaggle version

**Before running** (right sidebar -> Session options):
- Accelerator: **GPU T4 x2** or **GPU P100** (only one GPU is used).
- Internet: **On** (needs a phone-verified Kaggle account).

Output goes to `/kaggle/working/gita_audio` (verses, per-chapter wav+mp3, and a zip). Kaggle keeps `/kaggle/working` only when you **Save Version -> Save & Run All (Commit)**, which runs in the background without keeping the browser open (up to ~9-12 h). Download the files from the notebook's Output tab afterwards.

**Resuming after a stop:** the earlier output is not restored automatically. Add the previous run's output as an input (Add Input -> Notebook output / Your datasets) and set `PREV` in the *Prepare* cell; finished verses are copied in and skipped.

Kaggle's Python is newer than Vāgdhenu supports, so the model runs in a separate **Python 3.10 virtualenv** made with `uv` (`/tmp/venv`). 

Model: https://huggingface.co/prathoshap/vagdhenu (Apache-2.0; vocoder is a BigVGAN-v2 derivative, observe NVIDIA's terms; do not impersonate the voice). Text: https://github.com/gita/gita (check its license before redistributing the audio).

In [ ]:
!nvidia-smi -L

In [ ]:
%%bash
# One-time setup (~5-10 min). Mirrors scripts/setup.sh, inside a Python 3.10 venv.
set -e
pip install -q uv
cd /tmp
[ -d vagdhenu ] || git clone --depth 1 https://github.com/prathoshap/vagdhenu.git
cd vagdhenu
[ -d /tmp/venv ] || uv venv --python 3.10 /tmp/venv
PY=/tmp/venv/bin/python
TORCH="torch==2.4.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match"
uv pip install -q --python $PY $TORCH
uv pip install -q --python $PY -r requirements.txt
# requirements.txt can pull a newer torch/torchaudio (torchaudio>=2.9 needs torchcodec): re-pin the validated versions
uv pip install -q --python $PY $TORCH
$PY -c "import torch, torchaudio, numpy; print('torch', torch.__version__, 'torchaudio', torchaudio.__version__, 'numpy', numpy.__version__, 'cuda', torch.cuda.is_available())"
[ -d BigVGAN/.git ] || git clone --depth 1 https://github.com/NVIDIA/BigVGAN.git BigVGAN
PYTHONPATH="$PWD/BigVGAN" $PY scripts/download_weights.py
ls models
echo setup done

In [ ]:
# Prepare: paths, Gita text -> render jobs, and the render() helper. Re-run this cell after any restart.
import json, os, re, shutil, subprocess, urllib.request
from collections import Counter
import IPython.display as ipd

REPO = '/tmp/vagdhenu'
PY = '/tmp/venv/bin/python'
OUT = '/kaggle/working/gita_audio'
PREV = ''   # e.g. '/kaggle/input/gita-run1/gita_audio' to restore verses from an earlier run
os.chdir(REPO)
# MPLBACKEND=Agg: the notebook's inline backend does not exist inside the venv
ENV = dict(os.environ, PYTHONPATH=REPO + '/BigVGAN:' + REPO + '/src', MPLBACKEND='Agg')
os.makedirs(OUT + '/verses', exist_ok=True)
if PREV and os.path.isdir(PREV + '/verses'):
    n = 0
    for f in os.listdir(PREV + '/verses'):
        if not os.path.exists(f'{OUT}/verses/{f}'):
            shutil.copy(f'{PREV}/verses/{f}', f'{OUT}/verses/{f}'); n += 1
    print('restored', n, 'verse files from PREV')

url = 'https://raw.githubusercontent.com/gita/gita/main/data/verse.json'
verses = json.load(urllib.request.urlopen(url))
print(len(verses), 'verse records')

SPK = re.compile(r'^(धृतराष्ट्र उवाच|(?:सञ्जय|संजय) उवाच|अर्जुन उवाच|श्री ?भगवानुवाच)\s*')   # speaker (uvach) prefixes

def to_lines(text):
    lines = []
    for ln in text.split('\n'):
        ln = ln.strip()
        if not ln:
            continue
        ln = re.sub(r'।।\s*[\d.\-]+\s*।।', '', ln)   # verse-number marker
        ln = SPK.sub('', ln).strip()                  # drop a standalone speaker line, or the speaker glued to the verse
        ln = ln.rstrip('।॥').strip()
        if ln:
            lines.append(ln)
    return lines

shards = []
for v in verses:
    lines = to_lines(v['text'])
    if not lines:
        continue
    padas = [l + (' ॥' if i == len(lines) - 1 else ' ।') for i, l in enumerate(lines)]
    shards.append({'id': f"{v['chapter_number']:02d}_{v['verse_number']:03d}", 'meter': 'anuṣṭubh',
                   'padas': padas, 'seed': 60, 'no_sandhi': True})

# Meter detection runs inside the py3.10 venv (render_core needs torch). Falls back to anuṣṭubh on any error.
open('/tmp/detect.py', 'w').write('''
import json, sys
shards = json.load(open('/tmp/shards.json', encoding='utf8'))
try:
    from render_core import detect_meter_key
except Exception as e:
    print('detector unavailable:', e); sys.exit(0)
for s in shards:
    try:
        m = detect_meter_key(' '.join(s['padas']))
        if m: s['meter'] = m
    except Exception:
        pass
json.dump(shards, open('/tmp/shards.json', 'w', encoding='utf8'), ensure_ascii=False)
''')
json.dump(shards, open('/tmp/shards.json', 'w', encoding='utf8'), ensure_ascii=False)
r = subprocess.run([PY, '/tmp/detect.py'], env=ENV, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
shards = json.load(open('/tmp/shards.json', encoding='utf8'))
print(len(shards), 'verses;', Counter(s['meter'] for s in shards))

def render(jobs, tag):
    jobs = [j for j in jobs if not os.path.exists(f"{OUT}/verses/{j['id']}.wav")]
    if not jobs:
        return
    json.dump(jobs, open(f'/tmp/{tag}.json', 'w'), ensure_ascii=False)
    r = subprocess.run([PY, 'src/render.py', '--shard', f'/tmp/{tag}.json',
                        '--results', f'/tmp/{tag}_res.json', '--outdir', OUT + '/verses'],
                       env=ENV, capture_output=True, text=True)
    print('exit code', r.returncode); print((r.stdout or '')[-1500:])
    if r.returncode != 0:
        raise RuntimeError('render.py failed: ' + (r.stderr or '')[-2500:])

In [ ]:
# TEST FIRST: render chapter 1, verses 1-3 and listen. Expect '[batch] DONE 3/3 FAIL=0'.
import time; t = time.time()
render(shards[:3], 'test')
print('seconds:', round(time.time() - t))
for s in shards[:3]:
    p = f"{OUT}/verses/{s['id']}.wav"
    if os.path.exists(p): ipd.display(ipd.Audio(p))
    else: print('NOT CREATED:', p)

In [ ]:
# FULL RUN: all 18 chapters, resumable (skips existing wavs). Change the range to start elsewhere.
for ch in range(1, 19):
    jobs = [s for s in shards if s['id'].startswith(f'{ch:02d}_')]
    print('chapter', ch, len(jobs), 'verses')
    render(jobs, f'ch{ch:02d}')

In [ ]:
# Stitch verses into one audio file per chapter (+ mp3), then zip the mp3s
import numpy as np, soundfile as sf
os.makedirs(OUT + '/chapters', exist_ok=True)
SR, GAP = 24000, np.zeros(int(24000 * 1.2), dtype='float32')
missing = []
for ch in range(1, 19):
    parts = []
    for s in [s for s in shards if s['id'].startswith(f'{ch:02d}_')]:
        p = f"{OUT}/verses/{s['id']}.wav"
        if not os.path.exists(p):
            missing.append(s['id']); continue
        a, sr = sf.read(p, dtype='float32'); assert sr == SR, sr
        if a.ndim > 1: a = a.mean(1)
        parts += [a, GAP]
    if parts:
        wav = f'{OUT}/chapters/gita_ch{ch:02d}.wav'
        sf.write(wav, np.concatenate(parts), SR)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-b:a', '96k', wav[:-4] + '.mp3'])
print('missing verses (re-run the full-run cell):', missing)
if not missing:
    shutil.make_archive('/kaggle/working/gita_mp3', 'zip', OUT + '/chapters')  # includes wav+mp3; delete the wavs to shrink
    print('zip written: /kaggle/working/gita_mp3.zip')